[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/33_beam_search.ipynb)

# 🟠 Medium: Beam Search Decoding

Implement **beam search** — the classic decoding algorithm for sequence generation.

### Signature
```python
def beam_search(log_prob_fn, start_token, max_len, beam_width, eos_token) -> list[int]:
    # log_prob_fn: takes token list, returns (V,) log-probabilities
    # Returns: best sequence (list of ints)
```

### Algorithm
1. Start with `[(0.0, [start_token])]`
2. Each step: expand each beam with top-k next tokens
3. Keep top `beam_width` beams by total log-probability
4. Stop when best beam ends with `eos_token` or `max_len` reached

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [1]:
import torch

/usr/local/lib/python3.11/site-packages/torch/_subclasses/functional_tensor.py:307: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


In [9]:
# ✏️ YOUR IMPLEMENTATION HERE
def beam_search(log_prob_fn, start_token, max_len, beam_width, eos_token):
    # maintain beams, expand, prune, return best
    beams = [(0.0, [start_token])]
    for _ in range(max_len):
        candidates = []
        for score, seq in beams:
            if seq[-1] == eos_token:
                # if the sequence has ended
                candidates.append((score, seq))
                continue
            log_probs = log_prob_fn(seq)
            topk_values, topk_indices = torch.topk(log_probs, beam_width)
            for i in range(beam_width):
                # update candidates
                candidates.append((score + topk_values[i].item(), seq + [topk_indices[i].item()]))
        # select beam_width candidates
        candidates = sorted(candidates, key = lambda x:x[0], reverse=True)
        beams = candidates[:beam_width]
    best_score, best_seq = beams[0]
    return best_seq

In [10]:
# 🧪 Debug
def simple_fn(tokens):
    lp = torch.full((5,), -10.0)
    lp[min(len(tokens), 4)] = 0.0
    return lp

seq = beam_search(simple_fn, start_token=0, max_len=5, beam_width=2, eos_token=4)
print('Sequence:', seq)

Sequence: [0, 1, 2, 3, 4]


In [11]:
# ✅ SUBMIT
from torch_judge import check
check('beam_search')


🧪 Testing: Beam Search Decoding (Medium)
──────────────────────────────────────────────────
  ✅ [1/4] Returns list starting with start_token (1.4ms)
  ✅ [2/4] Greedy path (beam=1) (0.2ms)
  ✅ [3/4] Beam finds better path than greedy (0.2ms)
  ✅ [4/4] Stops at eos (0.2ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (2.0ms total)
  Progress saved. Run status() to see your dashboard.

